In [0]:
from pyspark.sql import functions as F

train = spark.table("databricks_gsales.silver.silver_train")
stores = spark.table("databricks_gsales.silver.silver_stores")

# 1. DAILY SALES
gold_daily_sales = (
    train.groupBy("date")
    .agg(
        F.sum("sales").alias("total_sales"),
        F.sum("onpromotion").alias("total_promoted_items"),
        F.countDistinct("store_nbr").alias("stores_count"),
        F.countDistinct("family").alias("product_families")
    )
)

gold_daily_sales.write.mode("overwrite").saveAsTable(
    "databricks_gsales.gold.gold_daily_sales"
)


# 2. STORE PERFORMANCE
gold_store_performance = (
    train.alias("t")
    .join(
        stores.alias("s"),
        F.col("t.store_nbr") == F.col("s.store_nbr"),
        "left"
    )
    .groupBy(
        F.col("t.store_nbr"),
        F.col("s.city"),
        F.col("s.state"),
        F.col("s.type")
    )
    .agg(
        F.sum("t.sales").alias("total_sales"),
        F.avg("t.sales").alias("avg_sales"),
        F.sum("t.onpromotion").alias("total_promoted_items")
    )
    .withColumnRenamed("type", "store_type")
)

gold_store_performance.write.mode("overwrite").saveAsTable(
    "databricks_gsales.gold.gold_store_performance"
)


# 3. FAMILY SALES
gold_family_sales = (
    train.groupBy("family")
    .agg(
        F.sum("sales").alias("total_sales"),
        F.avg("sales").alias("avg_sales"),
        F.sum("onpromotion").alias("total_promoted_items")
    )
)

gold_family_sales.write.mode("overwrite").saveAsTable(
    "databricks_gsales.gold.gold_family_sales"
)


# 4. PROMOTION ANALYSIS
promotion_df = train.withColumn(
    "promotion_status",
    F.when(F.col("onpromotion") > 0, "Promoted")
     .otherwise("Not Promoted")
)

gold_promotion_analysis = (
    promotion_df.groupBy("promotion_status")
    .agg(
        F.count("*").alias("record_count"),
        F.sum("sales").alias("total_sales"),
        F.avg("sales").alias("avg_sales"),
        F.sum("onpromotion").alias("total_promoted_items")
    )
)

gold_promotion_analysis.write.mode("overwrite").saveAsTable(
    "databricks_gsales.gold.gold_promotion_analysis"
)


# 5. MONTHLY SALES
gold_monthly_sales = (
    train
    .withColumn("sales_year", F.year("date"))
    .withColumn("sales_month", F.month("date"))
    .groupBy("sales_year", "sales_month")
    .agg(
        F.sum("sales").alias("total_sales"),
        F.avg("sales").alias("avg_sales"),
        F.sum("onpromotion").alias("total_promoted_items"),
        F.countDistinct("store_nbr").alias("stores_count")
    )
)

gold_monthly_sales.write.mode("overwrite").saveAsTable(
    "databricks_gsales.gold.gold_monthly_sales"
)


# 6. SALES SUMMARY
gold_sales_summary = (
    train.agg(
        F.sum("sales").alias("total_sales"),
        F.avg("sales").alias("avg_sales"),
        F.countDistinct("store_nbr").alias("total_stores"),
        F.countDistinct("family").alias("total_product_families"),
        F.sum("onpromotion").alias("total_promoted_items"),
        F.min("date").alias("start_date"),
        F.max("date").alias("end_date")
    )
)

gold_sales_summary.write.mode("overwrite").saveAsTable(
    "databricks_gsales.gold.gold_sales_summary"
)

print("All 6 Gold tables created successfully")

All 6 Gold tables created successfully
